<!-- Cell 1 -->
# AquaSelect -- Notebook 1a: ConvNeXt-Tiny Training on AQUA20

**Purpose**: Train ConvNeXt-Tiny backbone + linear classifier on AQUA20 across 3 seeds (9, 42, 2003).  
**Output**: 3 checkpoints + saved val/test logits for downstream selective prediction (Notebook 2).  
**Hardware**: Kaggle T4 GPU (16GB VRAM)  
**No selection head here** -- pure transfer learning baseline with cross-entropy loss.

In [ ]:
# Cell 2
!pip install -q timm datasets pyarrow

In [ ]:
# Cell 3
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms
from datasets import load_dataset, Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4
# Configuration 
CFG = {
    "model_name": "convnext_tiny",
    "num_classes": 20,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "seeds": [9, 42, 2003],
    # Phase 1: backbone frozen, classifier warmup
    "phase1_epochs": 5,
    "phase1_lr": 1e-3,
    # Phase 2: full fine-tuning
    "phase2_epochs": 45,       # total epochs = phase1 + phase2 = 50
    "phase2_lr": 1e-4,
    "weight_decay": 1e-4,
    "patience": 10,
    # Data split
    "val_size": 0.15,
    "split_seed": 42,          # fixed across all training seeds
    # Paths
    "data_path": "/kaggle/input/aqua20",
    "output_dir": "/kaggle/working",
}

print("Configuration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 5
# Load AQUA20 from parquet files 
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "/kaggle/input/datasets/abcd/aqua-20/train-00000-of-00001.parquet",
        "test":  "/kaggle/input/datasets/abcd/aqua-20/test-00000-of-00001.parquet",
    }
)

train_full = dataset["train"]
test_ds = dataset["test"]

print(f"Train (full): {len(train_full)} images")
print(f"Test:         {len(test_ds)} images")
print(f"Features:     {train_full.features}")
print(f"Label names:  {train_full.features['label'].names}")

In [ ]:
# Cell 6
# Stratified train/val split (fixed across all seeds) 
all_labels = train_full["label"]

train_indices, val_indices = train_test_split(
    range(len(train_full)),
    test_size=CFG["val_size"],
    random_state=CFG["split_seed"],
    stratify=all_labels,
)

print(f"Train subset: {len(train_indices)} images")
print(f"Val subset:   {len(val_indices)} images")
print(f"Test set:     {len(test_ds)} images")

# Verify stratification: compare class proportions
train_label_counts = pd.Series([all_labels[i] for i in train_indices]).value_counts().sort_index()
val_label_counts = pd.Series([all_labels[i] for i in val_indices]).value_counts().sort_index()
print(f"\nTrain class range: {train_label_counts.min()} - {train_label_counts.max()}")
print(f"Val class range:   {val_label_counts.min()} - {val_label_counts.max()}")

In [ ]:
# Cell 7
#  EDA: Class distribution 
label_names = train_full.features["label"].names

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, indices) in zip(axes, [
    ("Train", train_indices),
    ("Val", val_indices),
    ("Test", range(len(test_ds))),
]):
    if name == "Test":
        labels = test_ds["label"]
    else:
        labels = [all_labels[i] for i in indices]
    counts = pd.Series(labels).value_counts().sort_index()
    ax.barh(range(len(label_names)), [counts.get(i, 0) for i in range(len(label_names))], color="steelblue")
    ax.set_yticks(range(len(label_names)))
    ax.set_yticklabels(label_names, fontsize=9)
    ax.set_xlabel("Count")
    ax.set_title(f"{name} ({sum(counts)})")
    ax.invert_yaxis()

plt.suptitle("AQUA20 Class Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8
#  EDA: Sample images (2 per class) 

# Build a per-class index first to prevent scan sequentially
from collections import defaultdict
class_to_indices = defaultdict(list)
for i, lbl in enumerate(all_labels):
    if len(class_to_indices[lbl]) < 2:
        class_to_indices[lbl].append(i)

fig, axes = plt.subplots(4, 10, figsize=(20, 9))

for class_idx in range(CFG["num_classes"]):
    row_base = (class_idx // 10) * 2
    col = class_idx % 10
    for j, sample_idx in enumerate(class_to_indices[class_idx]):
        img = train_full[sample_idx]["image"]
        axes[row_base + j, col].imshow(img)
        axes[row_base + j, col].axis("off")
        if j == 0:
            axes[row_base + j, col].set_title(label_names[class_idx], fontsize=8, fontweight="bold")
    # Turn off axes for any unfilled slots
    for j in range(len(class_to_indices[class_idx]), 2):
        axes[row_base + j, col].axis("off")

plt.suptitle("AQUA20 Sample Images (2 per class)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9
# Dataset class and transforms 

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG["image_size"], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class AQUA20Dataset(Dataset):
    """Wraps a HuggingFace dataset split with optional index subsetting."""

    def __init__(self, hf_dataset, indices=None, transform=None):
        self.dataset = hf_dataset
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        item = self.dataset[self.indices[idx]]
        image = item["image"].convert("RGB")
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label


# Build datasets
train_dataset = AQUA20Dataset(train_full, train_indices, train_transform)
val_dataset   = AQUA20Dataset(train_full, val_indices,   eval_transform)
test_dataset  = AQUA20Dataset(test_ds,    None,           eval_transform)

# Build dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches x {CFG['batch_size']}")
print(f"Val loader:   {len(val_loader)} batches x {CFG['batch_size']}")
print(f"Test loader:  {len(test_loader)} batches x {CFG['batch_size']}")

#  check: load one batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}, Labels shape: {labels.shape}")
print(f"Label range: {labels.min().item()} - {labels.max().item()}")

In [ ]:
# Cell 10
# Model: ConvNeXt-Tiny backbone + linear classifier 

def create_model(model_name="convnext_tiny", num_classes=20):
    """Create a timm backbone with a linear classification head.
    
    Returns:
        model: nn.Module with .backbone and .classifier
        feature_dim: int, output dimension of the backbone
    """
    backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
    
    # Determine feature dim by a dummy forward pass
    with torch.no_grad():
        dummy = torch.randn(1, 3, 224, 224)
        feature_dim = backbone(dummy).shape[1]
    
    classifier = nn.Linear(feature_dim, num_classes)
    
    # Wrap into a single module
    class BackboneClassifier(nn.Module):
        def __init__(self, backbone, classifier):
            super().__init__()
            self.backbone = backbone
            self.classifier = classifier
        
        def forward(self, x):
            features = self.backbone(x)       # (B, feature_dim)
            logits = self.classifier(features) # (B, num_classes)
            return logits, features
    
    model = BackboneClassifier(backbone, classifier)
    print(f"Model: {model_name}")
    print(f"Feature dim: {feature_dim}")
    print(f"Backbone params: {sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M")
    print(f"Classifier params: {sum(p.numel() for p in classifier.parameters()) / 1e3:.1f}K")
    return model, feature_dim


# test
_model, _fdim = create_model(CFG["model_name"], CFG["num_classes"])
_dummy = torch.randn(2, 3, 224, 224)
_logits, _feats = _model(_dummy)
print(f"Logits shape: {_logits.shape}, Features shape: {_feats.shape}")
del _model, _dummy, _logits, _feats

In [ ]:
# Cell 11
# Utilities: seeding, metrics, training helpers 

def set_seed(seed):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_topk_accuracy(logits, labels, k=1):
    """Compute top-k accuracy from logits tensor and labels tensor."""
    topk_preds = logits.topk(k, dim=1).indices  # (N, k)
    correct = topk_preds.eq(labels.unsqueeze(1)).any(dim=1).float()
    return correct.mean().item() * 100


def evaluate(model, loader, criterion):
    """Run evaluation, return loss, top1, top3, macro_f1, all_logits, all_labels."""
    model.eval()
    total_loss = 0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    avg_loss = total_loss / len(all_labels)

    top1 = compute_topk_accuracy(all_logits, all_labels, k=1)
    top3 = compute_topk_accuracy(all_logits, all_labels, k=3)
    preds = all_logits.argmax(dim=1)
    macro_f1 = f1_score(all_labels.numpy(), preds.numpy(), average="macro") * 100

    return avg_loss, top1, top3, macro_f1, all_logits, all_labels


print("Utilities defined.")

In [ ]:
# Cell 12
# Training function: two-phase transfer learning

def train_one_seed(seed, train_loader, val_loader, cfg):
    """Train ConvNeXt-Tiny for one seed.
    
    Phase 1: Backbone frozen, train classifier only (5 epochs, lr=1e-3)
    Phase 2: Full fine-tuning with early stopping (up to 45 epochs, lr=1e-4)
    
    Returns:
        best_state: state_dict of best model (by val loss)
        history: dict with per-epoch metrics
        feature_dim: int
    """
    total_epochs = cfg["phase1_epochs"] + cfg["phase2_epochs"]
    
    print(f"\n{'='*70}")
    print(f"  SEED {seed} | Total epochs: {total_epochs} | Batch size: {cfg['batch_size']}")
    print(f"{'='*70}")
    
    set_seed(seed)
    model, feature_dim = create_model(cfg["model_name"], cfg["num_classes"])
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        "epoch": [], "phase": [],
        "train_loss": [], "val_loss": [],
        "val_top1": [], "val_top3": [], "val_f1": [],
    }
    
    global_epoch = 0
    seed_start_time = time.time()
    
    # Phase 1: Backbone frozen, classifier warmup 
    print(f"\n--- Phase 1: Classifier warmup (backbone frozen) ---")
    print(f"    Epochs: {cfg['phase1_epochs']} | LR: {cfg['phase1_lr']}")
    for param in model.backbone.parameters():
        param.requires_grad = False
    
    optimizer = AdamW(
        model.classifier.parameters(),
        lr=cfg["phase1_lr"], weight_decay=cfg["weight_decay"],
    )
    
    for epoch in range(1, cfg["phase1_epochs"] + 1):
        global_epoch += 1
        epoch_start = time.time()
        model.train()
        running_loss = 0
        correct = 0
        total = 0
        num_batches = len(train_loader)
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
            
            # Progress update every 20% of batches
            if (batch_idx + 1) % max(1, num_batches // 5) == 0 or (batch_idx + 1) == num_batches:
                pct = (batch_idx + 1) / num_batches * 100
                batch_loss = running_loss / total
                batch_acc = correct / total * 100
                print(f"    P1 Epoch {global_epoch}/{total_epochs} | "
                      f"Batch {batch_idx+1}/{num_batches} ({pct:.0f}%) | "
                      f"Loss: {batch_loss:.4f} | Acc: {batch_acc:.1f}%", end="\r")
        
        train_loss = running_loss / total
        train_acc = correct / total * 100
        val_loss, val_top1, val_top3, val_f1, _, _ = evaluate(model, val_loader, criterion)
        epoch_time = time.time() - epoch_start
        elapsed_total = time.time() - seed_start_time
        
        history["epoch"].append(global_epoch)
        history["phase"].append(1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_top1"].append(val_top1)
        history["val_top3"].append(val_top3)
        history["val_f1"].append(val_f1)
        
        overall_pct = global_epoch / total_epochs * 100
        print(f"\n  [{overall_pct:5.1f}%] P1 Epoch {global_epoch}/{total_epochs} "
              f"({epoch_time:.0f}s, total {elapsed_total/60:.1f}min) | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | "
              f"Val Loss: {val_loss:.4f} Top1: {val_top1:.1f}% Top3: {val_top3:.1f}% F1: {val_f1:.1f}%")
    
    # Phase 2: Full fine-tuning 
    print(f"\n--- Phase 2: Full fine-tuning ---")
    print(f"    Epochs: up to {cfg['phase2_epochs']} | LR: {cfg['phase2_lr']} | Patience: {cfg['patience']}")
    for param in model.backbone.parameters():
        param.requires_grad = True
    
    optimizer = AdamW(model.parameters(), lr=cfg["phase2_lr"], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["phase2_epochs"])
    
    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None
    best_epoch = 0
    
    for epoch in range(1, cfg["phase2_epochs"] + 1):
        global_epoch = cfg["phase1_epochs"] + epoch
        epoch_start = time.time()
        model.train()
        running_loss = 0
        correct = 0
        total = 0
        num_batches = len(train_loader)
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
            
            # Progress update every 20% of batches
            if (batch_idx + 1) % max(1, num_batches // 5) == 0 or (batch_idx + 1) == num_batches:
                pct = (batch_idx + 1) / num_batches * 100
                batch_loss = running_loss / total
                batch_acc = correct / total * 100
                print(f"    P2 Epoch {global_epoch}/{total_epochs} | "
                      f"Batch {batch_idx+1}/{num_batches} ({pct:.0f}%) | "
                      f"Loss: {batch_loss:.4f} | Acc: {batch_acc:.1f}%", end="\r")
        
        scheduler.step()
        train_loss = running_loss / total
        train_acc = correct / total * 100
        val_loss, val_top1, val_top3, val_f1, _, _ = evaluate(model, val_loader, criterion)
        epoch_time = time.time() - epoch_start
        elapsed_total = time.time() - seed_start_time
        
        # ETA calculation
        avg_epoch_time = elapsed_total / global_epoch
        epochs_remaining = cfg["phase2_epochs"] - epoch
        eta_min = avg_epoch_time * epochs_remaining / 60
        
        history["epoch"].append(global_epoch)
        history["phase"].append(2)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_top1"].append(val_top1)
        history["val_top3"].append(val_top3)
        history["val_f1"].append(val_f1)
        
        # Early stopping check
        improved = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = global_epoch
            improved = " ** BEST **"
        else:
            patience_counter += 1
        
        overall_pct = global_epoch / total_epochs * 100
        current_lr = scheduler.get_last_lr()[0]
        print(f"\n  [{overall_pct:5.1f}%] P2 Epoch {global_epoch}/{total_epochs} "
              f"({epoch_time:.0f}s, total {elapsed_total/60:.1f}min, ETA {eta_min:.1f}min) | "
              f"LR: {current_lr:.2e} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | "
              f"Val Loss: {val_loss:.4f} Top1: {val_top1:.1f}% Top3: {val_top3:.1f}% F1: {val_f1:.1f}% | "
              f"Patience: {patience_counter}/{cfg['patience']}{improved}")
        
        if patience_counter >= cfg["patience"]:
            print(f"\n  ** Early stopping at epoch {global_epoch} **")
            break
    
    total_time = (time.time() - seed_start_time) / 60
    print(f"\n  SEED {seed} COMPLETE | Best epoch: {best_epoch} | "
          f"Best val loss: {best_val_loss:.4f} | Total time: {total_time:.1f} min")
    print(f"{'='*70}")
    return best_state, history, feature_dim


print("Training function defined.")

In [ ]:
# Cell 13
# Train across all seeds 

all_results = {}   # seed -> { best_state, history, feature_dim }

total_start = time.time()

for seed in CFG["seeds"]:
    seed_start = time.time()
    best_state, history, feature_dim = train_one_seed(
        seed, train_loader, val_loader, CFG
    )
    elapsed = (time.time() - seed_start) / 60
    all_results[seed] = {
        "best_state": best_state,
        "history": history,
        "feature_dim": feature_dim,
    }
    print(f"\nSeed {seed} completed in {elapsed:.1f} min")

total_elapsed = (time.time() - total_start) / 60
print(f"\n{'='*60}")
print(f"All seeds completed in {total_elapsed:.1f} min")

In [ ]:
# Cell 14
# Training curves 

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for seed in CFG["seeds"]:
    h = all_results[seed]["history"]
    axes[0].plot(h["epoch"], h["train_loss"], label=f"Seed {seed} (train)")
    axes[0].plot(h["epoch"], h["val_loss"], "--", label=f"Seed {seed} (val)")
    axes[1].plot(h["epoch"], h["val_top1"], label=f"Seed {seed}")
    axes[2].plot(h["epoch"], h["val_f1"], label=f"Seed {seed}")

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Train / Val Loss"); axes[0].legend(fontsize=8)
axes[0].axvline(x=CFG["phase1_epochs"], color="gray", linestyle=":", alpha=0.5, label="Phase1/2 boundary")

axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Top-1 Accuracy (%)")
axes[1].set_title("Val Top-1 Accuracy"); axes[1].legend(fontsize=8)

axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Macro F1 (%)")
axes[2].set_title("Val Macro F1"); axes[2].legend(fontsize=8)

plt.suptitle("ConvNeXt-Tiny Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15
# Clean evaluation: reload best checkpoint, single forward pass on val + test 
# produces the definitive logits

criterion = nn.CrossEntropyLoss()

seed_metrics = []        # for summary table
saved_outputs = {}       # seed -> {val_logits, val_labels, test_logits, test_labels}

for seed in CFG["seeds"]:
    print(f"\n--- Clean eval for seed {seed} ---")
    
    # Rebuild model and load best state
    model, feature_dim = create_model(CFG["model_name"], CFG["num_classes"])
    model.load_state_dict(all_results[seed]["best_state"])
    model = model.to(device)
    model.eval()
    
    # Clean val pass
    val_loss, val_top1, val_top3, val_f1, val_logits, val_labels = evaluate(
        model, val_loader, criterion
    )
    print(f"  Val  | Loss: {val_loss:.4f} | Top1: {val_top1:.2f}% | Top3: {val_top3:.2f}% | F1: {val_f1:.2f}%")
    
    # Clean test pass
    test_loss, test_top1, test_top3, test_f1, test_logits, test_labels = evaluate(
        model, test_loader, criterion
    )
    print(f"  Test | Loss: {test_loss:.4f} | Top1: {test_top1:.2f}% | Top3: {test_top3:.2f}% | F1: {test_f1:.2f}%")
    
    seed_metrics.append({
        "seed": seed,
        "val_top1": val_top1, "val_top3": val_top3, "val_f1": val_f1,
        "test_top1": test_top1, "test_top3": test_top3, "test_f1": test_f1,
    })
    
    saved_outputs[seed] = {
        "val_logits": val_logits,
        "val_labels": val_labels,
        "test_logits": test_logits,
        "test_labels": test_labels,
    }
    
    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print("\nClean evaluation complete for all seeds.")

In [ ]:
# Cell 16
# Summary table: per-seed + mean/std + AQUA20 paper comparison 

df_metrics = pd.DataFrame(seed_metrics)

# Per-seed results
print("Per-seed test results:")
print(df_metrics.to_string(index=False))

# Mean +/- std
print(f"\n{'='*60}")
print("ConvNeXt-Tiny (Ours) -- Mean +/- Std across 3 seeds:")
print(f"  Top-1 Acc: {df_metrics['test_top1'].mean():.2f} +/- {df_metrics['test_top1'].std():.2f}%")
print(f"  Top-3 Acc: {df_metrics['test_top3'].mean():.2f} +/- {df_metrics['test_top3'].std():.2f}%")
print(f"  Macro F1:  {df_metrics['test_f1'].mean():.2f} +/- {df_metrics['test_f1'].std():.2f}%")

# Compare with AQUA20 paper baselines
print(f"\n{'='*60}")
print("Comparison with AQUA20 paper (Fuad et al., 2025):")
print(f"  ConvNeXt (paper):       Top1=90.69%  Top3=98.82%  F1=88.92%")
print(f"  ConvNeXt-Tiny (ours):   Top1={df_metrics['test_top1'].mean():.2f}%  "
      f"Top3={df_metrics['test_top3'].mean():.2f}%  "
      f"F1={df_metrics['test_f1'].mean():.2f}%")

In [ ]:
# Cell 17
# Per-class analysis: classification report + confusion matrix (seed 42) 

rep_seed = 42
rep_logits = saved_outputs[rep_seed]["test_logits"]
rep_labels = saved_outputs[rep_seed]["test_labels"]
rep_preds = rep_logits.argmax(dim=1)

# Classification report
print(f"Classification Report (seed {rep_seed}):\n")
print(classification_report(
    rep_labels.numpy(), rep_preds.numpy(),
    target_names=label_names, digits=3
))

# Confusion matrix
cm = confusion_matrix(rep_labels.numpy(), rep_preds.numpy())
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(CFG["num_classes"]))
ax.set_yticks(range(CFG["num_classes"]))
ax.set_xticklabels(label_names, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(label_names, fontsize=8)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix -- ConvNeXt-Tiny (Seed {rep_seed})")
plt.colorbar(im, ax=ax, shrink=0.8)

# Annotate cells
for i in range(CFG["num_classes"]):
    for j in range(CFG["num_classes"]):
        val = cm[i, j]
        if val > 0:
            color = "white" if val > cm.max() * 0.5 else "black"
            ax.text(j, i, str(val), ha="center", va="center", fontsize=7, color=color)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 18
# Save checkpoints + logits

output_dir = CFG["output_dir"]

# Save per-seed checkpoints
for seed in CFG["seeds"]:
    ckpt_path = os.path.join(output_dir, f"convnext_tiny_seed_{seed}.pth")
    torch.save({
        "model_state_dict": all_results[seed]["best_state"],
        "feature_dim": all_results[seed]["feature_dim"],
        "seed": seed,
        "model_name": CFG["model_name"],
        "num_classes": CFG["num_classes"],
    }, ckpt_path)
    print(f"Saved checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path) / 1e6:.1f} MB)")

# Save all val/test logits
logits_path = os.path.join(output_dir, "convnext_val_test_outputs.pth")
torch.save({
    "val_logits": {seed: saved_outputs[seed]["val_logits"] for seed in CFG["seeds"]},
    "val_labels": saved_outputs[CFG["seeds"][0]]["val_labels"],  # same for all seeds
    "test_logits": {seed: saved_outputs[seed]["test_logits"] for seed in CFG["seeds"]},
    "test_labels": saved_outputs[CFG["seeds"][0]]["test_labels"],  # same for all seeds
    "val_indices": val_indices,
    "train_indices": train_indices,
    "label_names": label_names,
    "config": CFG,
}, logits_path)
print(f"\nSaved logits: {logits_path} ({os.path.getsize(logits_path) / 1e6:.1f} MB)")

# Save metrics CSV
df_metrics.to_csv(os.path.join(output_dir, "convnext_seed_metrics.csv"), index=False)
print(f"Saved metrics CSV")

# List all output files
print(f"\nAll outputs in {output_dir}:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Cell 19
print("Notebook 1a complete.")